# Verkehrszeichen-Erkennung – Kaggle Training

**Voraussetzungen:**
1. Dieses Notebook in Kaggle öffnen
2. Rechts unter *Data* → *Add Data* → nach **`german-traffic-sign-detection-benchmark-gtsdb`** suchen (von safabouguezzi) → hinzufügen
3. Rechts unter *Settings* → **Accelerator: GPU T4 x2** aktivieren
4. Oben auf **Run All** klicken

Das Training dauert ca. 15–30 Minuten mit GPU.

In [ ]:
# 1. Repo klonen
!git clone https://github.com/XplorodoX/Verkehrschild-Erkennung.git
%cd Verkehrschild-Erkennung

In [ ]:
# 2. Abhängigkeiten installieren
!pip install -q ultralytics opencv-python-headless Pillow pandas plotly

In [ ]:
# 3. Kaggle-Datensatz-Pfad ermitteln
import os
from pathlib import Path

INPUT = Path('/kaggle/input')
gtsdb_root = None

for d in INPUT.iterdir():
    ppm_files = list(d.rglob('*.ppm'))
    gt_files  = list(d.rglob('gt.txt'))
    if ppm_files and gt_files:
        gtsdb_root = d
        print(f'GTSDB gefunden: {d}')
        print(f'  PPM-Bilder : {len(ppm_files)}')
        print(f'  gt.txt     : {gt_files[0]}')
        break

if gtsdb_root is None:
    raise RuntimeError(
        'Kein GTSDB-Datensatz gefunden.\n'
        'Bitte unter Data → Add Data nach '
        "'german-traffic-sign-detection-benchmark-gtsdb' suchen und hinzufügen."
    )

ppm_dir = ppm_files[0].parent
gt_file = gt_files[0]
print(f'\nBilder-Ordner : {ppm_dir}')
print(f'Annotationen  : {gt_file}')

In [ ]:
# 4. Datensatz konvertieren (PPM → JPEG, Annotationen → YOLO-Format)
import random
from pathlib import Path
from PIL import Image
import pandas as pd

IMG_TRAIN = Path('data/images/train')
IMG_VAL   = Path('data/images/val')
LBL_TRAIN = Path('data/labels/train')
LBL_VAL   = Path('data/labels/val')
for p in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    p.mkdir(parents=True, exist_ok=True)

VAL_SPLIT = 0.2
random.seed(42)

df = pd.read_csv(gt_file, sep=';', names=['filename','x1','y1','x2','y2','class_id'])
print(f'Annotationen: {len(df)} über {df["filename"].nunique()} Bilder')

all_images = sorted(ppm_dir.glob('*.ppm'))
print(f'PPM-Bilder  : {len(all_images)}')

shuffled  = all_images[:]
random.shuffle(shuffled)
train_set = set(img.name for img in shuffled[:int(len(shuffled) * (1 - VAL_SPLIT))])
annotations = df.groupby('filename')

for img_path in all_images:
    is_train = img_path.name in train_set
    img_dir  = IMG_TRAIN if is_train else IMG_VAL
    lbl_dir  = LBL_TRAIN if is_train else LBL_VAL
    dst_img  = img_dir / (img_path.stem + '.jpg')
    dst_lbl  = lbl_dir / (img_path.stem + '.txt')
    img      = Image.open(img_path).convert('RGB')
    img.save(dst_img, 'JPEG', quality=95)
    w, h = img.size
    if img_path.name in annotations.groups:
        lines = []
        for _, row in annotations.get_group(img_path.name).iterrows():
            xc = (row.x1 + row.x2) / 2 / w
            yc = (row.y1 + row.y2) / 2 / h
            bw = (row.x2 - row.x1) / w
            bh = (row.y2 - row.y1) / h
            lines.append(f'{int(row.class_id)} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
        dst_lbl.write_text('\n'.join(lines))
    else:
        dst_lbl.write_text('')

n_train = len(train_set)
n_val   = len(all_images) - n_train
print(f'\nTrain: {n_train} Bilder | Val: {n_val} Bilder – fertig!')

In [ ]:
# 5. Training starten (YOLOv8s, 100 Epochen, GPU)
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data='dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='verkehrszeichen_kaggle',
    patience=20,
    device=0,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    flipud=0.0,
    fliplr=0.0,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,
    degrees=12.0,
    translate=0.1,
    scale=0.6,
    shear=3.0,
    perspective=0.0003,
    mosaic=1.0,
    copy_paste=0.3,
    mixup=0.1,
    close_mosaic=10,
)

In [ ]:
# 6. Ergebnisse anzeigen
from ultralytics import YOLO
from pathlib import Path

best = Path('runs/detect/verkehrszeichen_kaggle/weights/best.pt')
model = YOLO(str(best))
metrics = model.val(data='dataset.yaml')

print('\n=== Ergebnisse ===')
print(f'  mAP50      : {metrics.box.map50:.3f}')
print(f'  mAP50-95   : {metrics.box.map:.3f}')
print(f'  Precision  : {metrics.box.mp:.3f}')
print(f'  Recall     : {metrics.box.mr:.3f}')

In [ ]:
# 7. Gewichte als ZIP speichern (zum Herunterladen)
import shutil

shutil.make_archive('/kaggle/working/best_weights', 'zip',
                    'runs/detect/verkehrszeichen_kaggle/weights')
print('Gespeichert: /kaggle/working/best_weights.zip')
print('Rechts unter Output → best_weights.zip herunterladen')